# Phase 3 — ML Model Development
### AI-Based Student Performance & Career Recommendation System
**University of South Asia, Lahore**

**Goal:**
- Linear Regression — grade prediction
- Random Forest Regressor — grade prediction
- Random Forest Classifier — Pass/Fail
- Cross Validation, Hyperparameter Tuning
- SHAP Values (Explainability)
- MLflow Tracking


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle, os, warnings, math

from sklearn.linear_model    import LinearRegression
from sklearn.ensemble        import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics         import (mean_squared_error, mean_absolute_error,
                                     r2_score, accuracy_score, f1_score,
                                     confusion_matrix, classification_report)
import shap
import mlflow
import mlflow.sklearn

warnings.filterwarnings('ignore')
os.makedirs('../plots',          exist_ok=True)
os.makedirs('../backend/models', exist_ok=True)
os.makedirs('../mlflow_tracking',exist_ok=True)

# Theme colors
BLUE  = '#2563EB'; CYAN  = '#06B6D4'
GREEN = '#10B981'; RED   = '#EF4444'
AMBER = '#F59E0B'; GRAY  = '#6B7280'

plt.rcParams['figure.facecolor'] = '#111827'
plt.rcParams['axes.facecolor']   = '#1F2937'
plt.rcParams['axes.edgecolor']   = '#374151'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['text.color']       = 'white'
plt.rcParams['grid.color']       = '#374151'
plt.rcParams['grid.alpha']       = 0.4

print("All imports successful!")


## Step 1 — Load Preprocessed Data

In [ ]:
splits = pickle.load(open('../data/processed/splits.pkl', 'rb'))

X_train   = splits['X_train'];   X_test   = splits['X_test']
y_train   = splits['y_train'];   y_test   = splits['y_test']
X_train_c = splits['X_train_c']; X_test_c = splits['X_test_c']
y_train_c = splits['y_train_c']; y_test_c = splits['y_test_c']
FEATURES  = splits['feature_names']

print("Data loaded successfully!")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features: {FEATURES}")


## Step 2 — Linear Regression (Grade Prediction)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

rmse_lr = math.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
r2_lr   = r2_score(y_test, y_pred_lr)

print("=== Linear Regression Results ===")
print(f"RMSE : {rmse_lr:.3f}")
print(f"MAE  : {mae_lr:.3f}")
print(f"R²   : {r2_lr:.3f}")
print()
print("Feature Coefficients:")
for feat, coef in sorted(zip(FEATURES, lr.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:20s}: {coef:+.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Linear Regression — Results', color='white', fontsize=13, fontweight='bold')

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_lr, color=BLUE, alpha=0.6, s=40)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             color=AMBER, linewidth=2, linestyle='--', label='Perfect line')
axes[0].set_xlabel('Actual Grade (%)')
axes[0].set_ylabel('Predicted Grade (%)')
axes[0].set_title(f'Actual vs Predicted  (R²={r2_lr:.3f})', color='white')
axes[0].legend()

# Residuals
residuals = y_test - y_pred_lr
axes[1].scatter(y_pred_lr, residuals, color=CYAN, alpha=0.6, s=40)
axes[1].axhline(0, color=AMBER, linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Grade (%)')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot', color='white')

plt.tight_layout()
plt.savefig('../plots/linear_regression_results.png', dpi=150,
            bbox_inches='tight', facecolor='#111827')
plt.show()
print("Saved: plots/linear_regression_results.png")


## Step 3 — Random Forest Regressor (Grade Prediction)

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)

rmse_rf = math.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
r2_rf   = r2_score(y_test, y_pred_rf)

print("=== Random Forest Regressor Results ===")
print(f"RMSE : {rmse_rf:.3f}")
print(f"MAE  : {mae_rf:.3f}")
print(f"R²   : {r2_rf:.3f}")


## Step 4 — Random Forest Classifier (Pass/Fail)

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train_c, y_train_c)
y_pred_clf = rf_clf.predict(X_test_c)

acc = accuracy_score(y_test_c, y_pred_clf)
f1  = f1_score(y_test_c, y_pred_clf)

print("=== Random Forest Classifier Results ===")
print(f"Accuracy : {acc:.3f}  ({acc*100:.1f}%)")
print(f"F1-Score : {f1:.3f}")
print()
print(classification_report(y_test_c, y_pred_clf,
                             target_names=['Fail', 'Pass']))


## Step 5 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_c, y_pred_clf)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fail', 'Pass'],
            yticklabels=['Fail', 'Pass'],
            linewidths=1, linecolor='#374151', ax=ax,
            annot_kws={'size': 14, 'weight': 'bold'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_title(f'Confusion Matrix  (Accuracy: {acc*100:.1f}%)',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/confusion_matrix.png', dpi=150,
            bbox_inches='tight', facecolor='#111827')
plt.show()
print("Saved: plots/confusion_matrix.png")


## Step 6 — Model Comparison (LR vs RF)

In [ ]:
comparison = pd.DataFrame({
    'Model':  ['Linear Regression', 'Random Forest'],
    'RMSE':   [round(rmse_lr, 3),   round(rmse_rf, 3)],
    'MAE':    [round(mae_lr,  3),   round(mae_rf,  3)],
    'R2':     [round(r2_lr,   3),   round(r2_rf,   3)],
})
print("=== Model Comparison ===")
print(comparison.to_string(index=False))
print()
if r2_rf > r2_lr:
    print(f"Winner: Random Forest (R2={r2_rf:.3f} vs {r2_lr:.3f})")
else:
    print(f"Winner: Linear Regression (R2={r2_lr:.3f} vs {r2_rf:.3f})")

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Comparison: Linear Regression vs Random Forest',
             color='white', fontsize=13, fontweight='bold')

metrics = [('RMSE (lower=better)', [rmse_lr, rmse_rf]),
           ('MAE  (lower=better)', [mae_lr,  mae_rf]),
           ('R²   (higher=better)',[r2_lr,   r2_rf])]

for ax, (title, vals) in zip(axes, metrics):
    bars = ax.bar(['Lin. Reg.', 'Rnd. Forest'], vals,
                  color=[BLUE, GREEN], alpha=0.85, width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.01,
                f'{v:.3f}', ha='center', color='white', fontsize=11)
    ax.set_title(title, color='white')

plt.tight_layout()
plt.savefig('../plots/model_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#111827')
plt.show()
print("Saved: plots/model_comparison.png")


## Step 7 — Cross Validation (5-Fold)

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

cv_lr = cross_val_score(LinearRegression(),
                         X_all, y_all, cv=5, scoring='r2')
cv_rf = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42),
                         X_all, y_all, cv=5, scoring='r2')

print("=== 5-Fold Cross Validation Results ===")
print(f"Linear Regression  R2 scores: {cv_lr.round(3)}")
print(f"  Mean: {cv_lr.mean():.3f}  |  Std: {cv_lr.std():.3f}")
print()
print(f"Random Forest      R2 scores: {cv_rf.round(3)}")
print(f"  Mean: {cv_rf.mean():.3f}  |  Std: {cv_rf.std():.3f}")

# CV plot
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(5)
width = 0.35
bars1 = ax.bar(x - width/2, cv_lr, width, label='Linear Regression',
               color=BLUE, alpha=0.85)
bars2 = ax.bar(x + width/2, cv_rf, width, label='Random Forest',
               color=GREEN, alpha=0.85)
ax.axhline(cv_lr.mean(), color=BLUE,  linestyle='--', alpha=0.6,
           label=f'LR Mean: {cv_lr.mean():.3f}')
ax.axhline(cv_rf.mean(), color=GREEN, linestyle='--', alpha=0.6,
           label=f'RF Mean: {cv_rf.mean():.3f}')
ax.set_xlabel('Fold')
ax.set_ylabel('R² Score')
ax.set_title('5-Fold Cross Validation', color='white', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Fold 1','Fold 2','Fold 3','Fold 4','Fold 5'])
ax.legend()
plt.tight_layout()
plt.savefig('../plots/cross_validation.png', dpi=150,
            bbox_inches='tight', facecolor='#111827')
plt.show()
print("Saved: plots/cross_validation.png")


## Step 8 — Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 10, 20],
    'min_samples_split': [2, 5],
}
print("Running GridSearchCV... (this may take a minute)")

grid = GridSearchCV(RandomForestRegressor(random_state=42),
                    param_grid, cv=3, scoring='r2',
                    n_jobs=-1, verbose=0)
grid.fit(X_train, y_train)
best_rf = grid.best_estimator_

y_pred_best = best_rf.predict(X_test)
rmse_best   = math.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best     = r2_score(y_test, y_pred_best)

print(f"Best Params: {grid.best_params_}")
print(f"Best CV R²:  {grid.best_score_:.3f}")
print()
print(f"Before tuning — RMSE: {rmse_rf:.3f}  R2: {r2_rf:.3f}")
print(f"After tuning  — RMSE: {rmse_best:.3f}  R2: {r2_best:.3f}")


## Step 9 — SHAP Values (Explainability)

In [ ]:
print("Calculating SHAP values...")
explainer   = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test,
                  feature_names=FEATURES,
                  show=False,
                  plot_type='bar')
plt.title('SHAP Feature Importance', color='white',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/shap_summary.png', dpi=150,
            bbox_inches='tight', facecolor='#111827')
plt.show()
print("Saved: plots/shap_summary.png")


In [ ]:
# SHAP for single student (index 0)
import shap, matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 4))
exp = shap.Explanation(
    values        = shap_values[0].astype(float),
    base_values   = float(explainer.expected_value),
    data          = X_test[0].astype(float),
    feature_names = FEATURES
)
shap.plots.waterfall(exp, show=False)
plt.title(f"Single Student SHAP — Predicted: {y_pred_best[0]:.1f}%",
          color="white", fontsize=12)
plt.tight_layout()
plt.savefig("../plots/shap_waterfall_single.png", dpi=150,
            bbox_inches="tight", facecolor="#111827")
plt.show()
print("Saved: plots/shap_waterfall_single.png")

print("
SHAP impact for student #1:")
for feat, sv in sorted(zip(FEATURES, shap_values[0]),
                        key=lambda x: abs(x[1]), reverse=True):
    direction = "positive" if sv > 0 else "negative"
    print(f"  {feat:20s}: {sv:+.3f}  ({direction} impact)")

## Step 10 — MLflow Experiment Tracking

In [ ]:
mlflow.set_tracking_uri('file:../mlflow_tracking')
mlflow.set_experiment('Student_Performance_Prediction')

# Log Linear Regression
with mlflow.start_run(run_name='Linear_Regression'):
    mlflow.log_param('model_type', 'LinearRegression')
    mlflow.log_metric('RMSE', rmse_lr)
    mlflow.log_metric('MAE',  mae_lr)
    mlflow.log_metric('R2',   r2_lr)
    mlflow.sklearn.log_model(lr, 'linear_regression_model')
    mlflow.log_artifact('../plots/linear_regression_results.png')
    print("Logged: Linear Regression")

# Log Random Forest (default)
with mlflow.start_run(run_name='Random_Forest_Default'):
    mlflow.log_param('model_type',   'RandomForest')
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('tuned',        False)
    mlflow.log_metric('RMSE', rmse_rf)
    mlflow.log_metric('MAE',  mae_rf)
    mlflow.log_metric('R2',   r2_rf)
    mlflow.sklearn.log_model(rf_reg, 'random_forest_model')
    print("Logged: Random Forest Default")

# Log Random Forest (tuned)
with mlflow.start_run(run_name='Random_Forest_Tuned_Best'):
    mlflow.log_params(grid.best_params_)
    mlflow.log_param('model_type', 'RandomForest_Tuned')
    mlflow.log_param('tuned',      True)
    mlflow.log_metric('RMSE',      rmse_best)
    mlflow.log_metric('R2',        r2_best)
    mlflow.log_metric('CV_R2_mean',grid.best_score_)
    mlflow.log_metric('Accuracy',  acc)
    mlflow.log_metric('F1_Score',  f1)
    mlflow.sklearn.log_model(best_rf, 'best_rf_model')
    mlflow.log_artifact('../plots/shap_summary.png')
    mlflow.log_artifact('../plots/confusion_matrix.png')
    mlflow.log_artifact('../plots/model_comparison.png')
    print("Logged: Random Forest Tuned (Best)")

print()
print("MLflow tracking complete!")
print("Run 'mlflow ui --backend-store-uri mlflow_tracking' to view dashboard")


## Step 11 — Save All Models

In [ ]:
pickle.dump(lr,       open('../backend/models/linear_regression.pkl','wb'))
pickle.dump(best_rf,  open('../backend/models/random_forest.pkl',    'wb'))
pickle.dump(rf_clf,   open('../backend/models/rf_classifier.pkl',    'wb'))

print("=== Models Saved ===")
for f in os.listdir('../backend/models'):
    size = os.path.getsize(f'../backend/models/{f}')
    print(f"  {f:35s}  {size/1024:.1f} KB")


In [ ]:
print("=" * 55)
print("PHASE 3 COMPLETE — FINAL RESULTS SUMMARY")
print("=" * 55)
print()
print("REGRESSION (Grade Prediction)")
print(f"  Linear Regression  — RMSE:{rmse_lr:.2f}  MAE:{mae_lr:.2f}  R2:{r2_lr:.3f}")
print(f"  Random Forest      — RMSE:{rmse_rf:.2f}  MAE:{mae_rf:.2f}  R2:{r2_rf:.3f}")
print(f"  RF Tuned (Best)    — RMSE:{rmse_best:.2f}  R2:{r2_best:.3f}")
print()
print("CLASSIFICATION (Pass/Fail)")
print(f"  Random Forest      — Accuracy:{acc*100:.1f}%  F1:{f1:.3f}")
print()
print("Cross Validation (5-Fold)")
print(f"  LR  Mean R2: {cv_lr.mean():.3f}")
print(f"  RF  Mean R2: {cv_rf.mean():.3f}")
print()
print("Best Model: Random Forest (Tuned)")
print(f"  Best Params: {grid.best_params_}")
print()
print("Plots saved:  plots/")
print("Models saved: backend/models/")
print("MLflow logs:  mlflow_tracking/")
print()
print("Next: Phase 4 — FastAPI Backend")


## Phase 3 Summary

| Model | RMSE | R² | Notes |
|---|---|---|---|
| Linear Regression | - | - | Baseline model |
| Random Forest (default) | - | - | Better than LR |
| Random Forest (tuned) | - | - | Best model |
| RF Classifier (Pass/Fail) | - | Acc/F1 | Classification |

**All models saved in `backend/models/`**
**Next: Phase 4 — FastAPI Backend (04_fastapi.ipynb)**
